# Druksensor ijken en maken van een $pV$-diagram

## Introductie

In de experimentele natuurkunde was het lang geleden gelukt om de krachten tussen ladingen te bestuderen zonder dat bekend was hoe groot die ladingen nu precies waren. Men laadde een metalen bol op en hield deze tegen een andere metalen bol van hetzelfde materiaal. Men redeneerde dat de ladingen op de bollen gelijk waren, omdat ze van hetzelfde materiaal waren. Vervolgens plaatste men de bollen in een vacuüm en mat men de krachten tussen de bollen met een zeer gevoelige balans. Op deze manier kon men de krachten tussen de ladingen bestuderen zonder de absolute waarde van de ladingen te kennen. Dit trucje kon herhaald worden met andere bollen waarna een kwantiatieve beschrijving van de krachten tussen ladingen mogelijk werd.

Een soortgelijke meettechniek gaan we gebruiken om een druksensor te ijken. Van de sensor zijn wel wat dingen bekend, maar omdat de spanning van de Arduino niet overeenkomstig is met de gewenste spanning, zouden we deze moeten ijken. We weten dat de sensor lineair is, dus als we twee punten weten, kunnen we de rest van de curve bepalen. Nog beter zou het zijn om drie punten te nemen en zo ook het lineaire karakter van de sensor te bevestigen.

## Theorie

Een injectiespuit met een maximaal volume van 50 mL is gevuld met lucht. De spuit kan aan een kant afgesloten worden met een tube die verbonden is met een druksensor die de gasdruk meet. Door de zuiger van de spuit in te drukken, wordt het volume verkleind en de druk verhoogd. Wanneer we de druk langzaam in drukken verwachten we dat de druk in de spuit volgens de wet van Boyle toeneemt:

$$
    P_1 V_1 = P_2 V_2 
$$ (eq:Boyle)

Omdat de gemeten spanning van de druksensor lineair afhankelijk is van de druk, kan de druk uitgedrukt worden als:

$$
    P = a U + b
$$ (eq:lineair)



## Methode en materialen

```{note} Software
De Arduino code staat al op de Arduino's. Als je de Arduino aansluit op je computer en de Arduino IDE opent, kan je de seriële monitor openen om de gemeten spanning te zien.
```

Je maakt gebruik van een Arduino. Daarvoor heb je de juiste IDE nodig. Het programma staat al op de Arduino's in het lokaal. Zodra je de Arduino aansluit op je computer zal de Arduino gaan meten, maar zijn de metingen nog niet zichtbaar. Je moet de Arduino op `Arduino MKR Zero` zetten. Dan wordt nog wel een driver geinstalleerd. 

Controleer of de Arduino herkend wordt door op `tools` -> `port` te klikken, daar staat de com poort van de Arduino. Open vervolgens de seriële monitor (het vergrootglas rechtsboven in de IDE) om de gemeten spanning te zien.

```{warning}
De twee stekkertjes hoef je NIET met elkaar te verbinden. Dit is alleen voor een meting in de brandblusser.
``` 

```{code} C++
int drukpin = A1;

void setup() {
  pinMode(A1,INPUT);
  Serial.begin(9600);
}

void loop() {
  Serial.println(analogRead(drukpin));
  delay(100);
}
```

### Deel 1
Stel de injectiespuit in op 40 mL en sluit de spuit aan op de druksensor door middel van een zo klein mogelijke tube. Meet de spanning van de druksensor met de Arduino en noteer deze waarde als $U_1$. Druk vervolgens de zuiger langzaam in tot 20 mL en meet opnieuw de spanning van de druksensor, noteer deze waarde als $U_2$. Herhaal dit voor volumes van 10 mL. 

1. Leg uit waarom een zo klein mogelijke tube gebruikt moet worden.
2. Welke waarde hoort bij de gasdruk bij 40 mL? Zoek deze waarde op.
3. Welke waarden horen bij de gasdruk bij 20 en 10 mL? 
4. Gebruik de drie punten om de waarden van $a$ en $b$ in {numref}`vergelijking {number} <eq:lineair>` te bepalen en controleer of de sensor inderdaad lineair is door de waarden te plotten.

In [1]:
#Import blah blah
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [2]:
#Raw data importeren
VData1 = np.array([40,20,10])        #Volume injectiespuit meting1(mL)
UData1 = np.array([178,307,508])     #Gemeten spaning Arduino meting1 (V)

def func1(a,U,b):       #Defineren van linear functie
    return a * U + b

#Fitten van meting1
popt1, pcov1 = curve_fit(func1, VData1, UData1)
a_fit1, b_fit1 = popt1

PData1 = func1(a_fit1,UData1,b_fit1)        #P volgens fit en meetdata1

#Trendlijn van meting1
U_fit1 = np.linspace(min(UData1),max(UData1),1000)
P_fit1 = func1(a_fit1,U_fit1,b_fit1)

### Deel 2
Vervang daarbij de kleine tube voor een langere en bepaal het onbekende volume van de tube met een volgende meetserie waarbij je de druk en het volume bepaald. Zorg ervoor dat ook drukken onder de 1 atm gemeten worden. 
```{tip}
Maak gebruik van een systematische fout in het volume om het volume van de tube te vinden.
```

## Resultaten

In [3]:
### Jouw data en code
VData2 = np.array([60,50,40,30,20,10])           #Volume injectiespuit meting2(mL)
UData2 = np.array([137,155,180,223,296,467])     #Gemeten spanning Arduino meting2 (V)  

def func2(a,U,b):
    return a * U + b

#Fitten van meting2
popt2, pcov2 = curve_fit(func2, VData2, UData2)
a_fit2, b_fit2 = popt2

#Gemiddelde helling van beide plots
a = (a_fit1 + a_fit2) / 2

#Verschil in volume van korte en lange buis (b)
d_b = b_fit2 - b_fit1

#Uitrekenen volume van lange buis
V_buis = d_b / a

print("Tube volume difference =", V_buis, "mL")


Tube volume difference = 14.381807092445777 mL


In [4]:
#Gezamelijk arrays met index array voor korte en lange buis
V_all = np.concatenate([VData1,VData2])
U_all = np.concatenate([UData1,UData2])
index = np.concatenate([np.zeros_like(VData1), np.ones_like(VData2)])

#Functie U = aV + b1 voor korte buis en b2 voor lange buis
def func(x,a,b1,b2):
    V, index = x
    return a * V + np.where(index < 0.5, b1, b2)

popt, pcov = curve_fit(func, (V_all, index), U_all)
a, b1, b2 = popt
perr = np.sqrt(np.diag(pcov))

d_b = b2 - b1
V_t = d_b / a

d_da = -d_b / (a**2)
d_db1 = -1.0 / a
d_db2 =  1.0 / a
J = np.array([d_da, d_db1, d_db2])

print("a = {:.4f} ± {:.4f}".format(a, perr[0]))
print("b1 = {:.3f} ± {:.3f}  (korte buis)".format(b1, perr[1]))
print("b2 = {:.3f} ± {:.3f}  (lange buis)".format(b2, perr[2]))
print("V van buis = {:.3f}  (mL)".format(V_t))



a = -6.9519 ± 1.3578
b1 = 493.211 ± 48.641  (korte buis)
b2 = 486.316 ± 54.217  (lange buis)
V van buis = 0.992  (mL)
